In [1]:
# installing necessary libraries
!pip install scikit-learn sweetviz

In [2]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn. linear_model import LinearRegression

In [3]:
# loading the dataset and initial EDA with SweetViz
insurance = pd.read_csv('insurance.csv')
insurance.head()

# visualizing the data with SweetViz
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


The main trends and patterns:
- It seems that there is a high correlation between smoking and charges.
- It also seems that there is a correlation between age and charges, and region and bmi.
- For the most part, the data is split evenly between males and females.
- Most adults have 0 children.
- The Southeast region has the highest number of people.
- Most people in this dataset are non-smokers.
- Age and bmi are normally distributed.

In [4]:
#encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()


,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [5]:
# preparing the data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scaling the features (don't need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#baseline linear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#baseline random forest model
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf) 

# print baseline results
print(f'Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest R2: {mse_rf:.2f}, R2: {r2_rf:.2f}')

Baseline Linear Regression MSE: 33566439.74, R2: 0.78
Baseline Random Forest R2: 22179121.67, R2: 0.86


In [6]:
# CV with baseline models with r2 as the scoring metric
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf}')   


Baseline Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Baseline Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


The linear regression model is underfitting on the test data. It is some pf the missing relationships that are being predicted. This model has a higher MSE and lower R^2.

The random forest model performs better on the test data. It seems to be improving model performance and reducing variance. This is probably based on the randomly selected subsets, which are based on the best split. This can also be seen with the lower MSE and higher R^2. 

In [7]:
# gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['log2', 'sqrt']
}           

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)    
print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')


Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV R2 Score: 0.84


In [8]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
11           200        10                  5         log2         0.841202
3            200      None                  5         log2         0.840662
19           200        20                  5         log2         0.840662
10           100        10                  5         log2         0.840452
8            100        10                  2         log2         0.840383
9            200        10                  2         log2         0.839967
2            100      None                  5         log2         0.839195
18           100        20                  5         log2         0.839195
17           200        20                  2         log2         0.836639
1            200      None                  2         log2         0.836593
16           100        20                  2         log2         0.835721
0            100      None                  2         log2

I would choose the model with the following hyperparameters:
- n_estimators: I would choose 200. I think having more trees would reduce variance and give better predictions.
- max_depth: I would choose 20. A moderate max depth controls overfitting and also allows for complex interactions.
- min_samples_split: I would choose 2 because it keeps trees flexible.
- max_features: I would choose sqrt because there isn't much difference between log2 and it is a strong default choice. It reduces correlation and improves generalization between trees.

In [9]:
# Evaluate tuned RandomForest on training and test sets using the best estimator from GridSearchCV
best_rf = grid_search_rf.best_estimator_
# predictions
train_pred = best_rf.predict(X_train)
test_pred = best_rf.predict(X_test)
# metrics
train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
print(f'Training MSE: {train_mse:.2f}, R2: {train_r2:.3f}')
print(f'Test     MSE: {test_mse:.2f}, R2: {test_r2:.3f}')

Training MSE: 10067783.72, R2: 0.930
Test     MSE: 20430805.91, R2: 0.868


Comparing training vs test performance:

I believe the model is overfitting the data. It seems to have memorized the training data. The training R^2 score is much higher than the R^2 test score. In addition, the MSE training score is much lower than the MSE testing score. More tuning would need to be done in order to improve testing on the test data.